In [ ]:
import pandas as pd
import numpy as np


In [ ]:
df = pd.read_parquet('bureau.parquet')
print('Shape:', df.shape)
print('Уникальных клиентов (SK_ID_CURR):', df['SK_ID_CURR'].nunique())
print('Уникальных кредитов (SK_ID_BUREAU):', df['SK_ID_BUREAU'].nunique())
print('\nПропуски топ-20:')
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(missing[missing > 0].head(20))
print('\nЧисло категориальных фич:')
print(df.select_dtypes(include='object').shape[1])
print('\nЧисло числовых фич:')
print(df.select_dtypes(include='number').shape[1])

Группы фичей с большими пропусками + категориальные фичи и их уникальные значения \
Порог для «высоких» пропусков — 40%

In [ ]:
high_missing = missing[missing > 40].index.tolist()
print(f'Фичи с пропусками >40%: {len(high_missing)}')
print(high_missing)

print('\nКатегориальные фичи:')
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    print(f'  {col}: {df[col].nunique()} уникальных')
    print(df[col].value_counts().to_string())
    print()

In [ ]:
# Числовые фичи — describe
num_cols = df.select_dtypes(include='number').columns.drop(['SK_ID_CURR', 'SK_ID_BUREAU'])
print(f'Числовых фич (без ID): {len(num_cols)}')
print(df[num_cols].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(2))

In [ ]:
# Выбросы — перцентили 99% и 99.9%
print('Перцентили ключевых числовых фич:')
for col in num_cols:
    q99  = df[col].quantile(0.99)
    q999 = df[col].quantile(0.999)
    maxv = df[col].max()
    print(f'  {col}: 99%={q99:.1f}, 99.9%={q999:.1f}, max={maxv:.1f}')

## Очистка

1. `CREDIT_TYPE` — редкие типы (<1000 записей) → `'Other'`  
2. `CREDIT_ACTIVE` — `'Bad debt'` (21 шт) и `'Sold'` (6527 шт) → `'Other'`  
3. `CREDIT_CURRENCY` — 99.9% = `currency 1`, остальные игнорируем при агрегации  
4. Клипинг выбросов `AMT_CREDIT_SUM`, `AMT_CREDIT_MAX_OVERDUE`, `AMT_CREDIT_SUM_DEBT` на уровне 99.9%

In [ ]:
# 1. Группировка редких CREDIT_TYPE
rare_types = df['CREDIT_TYPE'].value_counts()
rare_types = rare_types[rare_types < 1000].index.tolist()
df['CREDIT_TYPE_GROUPED'] = df['CREDIT_TYPE'].apply(
    lambda x: 'Other' if x in rare_types else x
)
print('CREDIT_TYPE_GROUPED:')
print(df['CREDIT_TYPE_GROUPED'].value_counts())

# 2. Группировка CREDIT_ACTIVE
df['CREDIT_ACTIVE_GROUPED'] = df['CREDIT_ACTIVE'].apply(
    lambda x: 'Other' if x in ['Bad debt', 'Sold'] else x
)
print('\nCREDIT_ACTIVE_GROUPED:')
print(df['CREDIT_ACTIVE_GROUPED'].value_counts())

# 3. Клипинг выбросов
clip_cols = ['AMT_CREDIT_SUM', 'AMT_CREDIT_MAX_OVERDUE', 'AMT_CREDIT_SUM_DEBT']
for col in clip_cols:
    cap = df[col].quantile(0.999)
    df[col] = df[col].clip(upper=cap)
    print(f'{col} clipped at 99.9% = {cap:.0f}')

## Агрегация на уровень SK_ID_CURR  
Предполагается, что `bureau_balance_EDA.ipynb` уже выполнен и файл `bb_agg.csv` сохранён.

In [ ]:
# Подгружаем агрегаты из bureau_balance (результат bureau_balance_EDA.ipynb)
bb_agg = pd.read_csv('bb_agg.csv')
df = df.merge(bb_agg, on='SK_ID_BUREAU', how='left')
print('bureau shape после мерджа с bb_agg:', df.shape)
print(f"Кредитов с историей bb: {df['bb_count'].notna().sum()} / {len(df)} ({df['bb_count'].notna().mean()*100:.1f}%)")

In [ ]:
bureau_agg = df.groupby('SK_ID_CURR').agg(
    # Кол-во кредитов
    bureau_loan_count        = ('SK_ID_BUREAU',          'count'),
    bureau_active_count      = ('CREDIT_ACTIVE_GROUPED', lambda x: (x == 'Active').sum()),
    bureau_closed_count      = ('CREDIT_ACTIVE_GROUPED', lambda x: (x == 'Closed').sum()),
    # Временные признаки
    bureau_days_credit_mean  = ('DAYS_CREDIT',           'mean'),
    bureau_days_credit_min   = ('DAYS_CREDIT',           'min'),
    bureau_enddate_max       = ('DAYS_CREDIT_ENDDATE',   'max'),
    bureau_enddate_mean      = ('DAYS_CREDIT_ENDDATE',   'mean'),
    # Просрочки (из bureau)
    bureau_overdue_max       = ('CREDIT_DAY_OVERDUE',    'max'),
    bureau_overdue_mean      = ('CREDIT_DAY_OVERDUE',    'mean'),
    bureau_overdue_sum       = ('CREDIT_DAY_OVERDUE',    'sum'),
    # Суммы
    bureau_amt_credit_sum    = ('AMT_CREDIT_SUM',        'sum'),
    bureau_amt_credit_mean   = ('AMT_CREDIT_SUM',        'mean'),
    bureau_amt_credit_max    = ('AMT_CREDIT_SUM',        'max'),
    bureau_amt_debt_sum      = ('AMT_CREDIT_SUM_DEBT',   'sum'),
    bureau_amt_debt_mean     = ('AMT_CREDIT_SUM_DEBT',   'mean'),
    bureau_amt_overdue_max   = ('AMT_CREDIT_MAX_OVERDUE','max'),
    bureau_amt_overdue_sum   = ('AMT_CREDIT_SUM_OVERDUE','sum'),
    # Пролонгации
    bureau_prolong_sum       = ('CNT_CREDIT_PROLONG',    'sum'),
    bureau_prolong_max       = ('CNT_CREDIT_PROLONG',    'max'),
    # Аннуитет
    bureau_annuity_sum       = ('AMT_ANNUITY',           'sum'),
    bureau_annuity_mean      = ('AMT_ANNUITY',           'mean'),
    # Дата обновления
    bureau_days_update_mean  = ('DAYS_CREDIT_UPDATE',    'mean'),
    bureau_days_update_max   = ('DAYS_CREDIT_UPDATE',    'max'),
    # Из bureau_balance
    bureau_bb_dpd_max        = ('bb_dpd_max',            'max'),
    bureau_bb_dpd_mean       = ('bb_dpd_mean',           'mean'),
    bureau_bb_overdue_any    = ('bb_overdue_any',        'max'),
    bureau_bb_months_min     = ('bb_months_min',         'min'),
).reset_index()

# Производные фичи
bureau_agg['bureau_active_ratio']      = bureau_agg['bureau_active_count'] / bureau_agg['bureau_loan_count']
bureau_agg['bureau_debt_credit_ratio'] = bureau_agg['bureau_amt_debt_sum'] / (bureau_agg['bureau_amt_credit_sum'] + 1)

print('Shape bureau_agg:', bureau_agg.shape)
print(f'Фич для джойна: {bureau_agg.shape[1] - 1}')

In [ ]:
print('Пропуски в bureau_agg (%):')
missing_agg = (bureau_agg.isnull().sum() / len(bureau_agg) * 100).sort_values(ascending=False)
print(missing_agg[missing_agg > 0])

# Заполняем NaN нулями
fill_zero_cols = [
    'bureau_annuity_sum', 'bureau_annuity_mean',
    'bureau_bb_dpd_max', 'bureau_bb_dpd_mean',
    'bureau_bb_overdue_any', 'bureau_bb_months_min',
    'bureau_amt_overdue_max', 'bureau_amt_overdue_sum',
    'bureau_amt_debt_sum', 'bureau_amt_debt_mean',
]
bureau_agg[fill_zero_cols] = bureau_agg[fill_zero_cols].fillna(0)

print('\nПосле fillna(0):')
missing_final = (bureau_agg.isnull().sum() / len(bureau_agg) * 100).sort_values(ascending=False)
print(missing_final[missing_final > 0] if missing_final[missing_final > 0].any() else 'Все пропуски заполнены!')

bureau_agg.to_csv('bureau_agg.csv', index=False)
print(f'\nСохранено: bureau_agg.csv  —  {bureau_agg.shape[0]} строк, {bureau_agg.shape[1]} колонок')

In [ ]:
# Топ-15 по корреляции с TARGET
app = pd.read_csv('application_train_all_models.csv', usecols=['SK_ID_CURR', 'TARGET'], index_col=0)
merged = app.merge(bureau_agg, on='SK_ID_CURR', how='left')

num_corr = merged.select_dtypes(include='number').corr()['TARGET'].abs().sort_values(ascending=False)
print('Топ-15 фич bureau по |корреляции| с TARGET:')
print(num_corr.head(16))

In [ ]:
# Джойн к основному датасету
app_train = pd.read_csv('application_train_all_models.csv', index_col=0)
print('Shape до джойна:', app_train.shape)

app_train = app_train.merge(bureau_agg, on='SK_ID_CURR', how='left')
bureau_feature_cols = [c for c in bureau_agg.columns if c != 'SK_ID_CURR']
app_train[bureau_feature_cols] = app_train[bureau_feature_cols].fillna(0)

print('Shape после джойна:', app_train.shape)
covered = (app_train['bureau_loan_count'] > 0).sum()
print(f"Клиентов с историей в ЦКИ: {covered} ({covered/len(app_train)*100:.1f}%)")

app_train.to_csv('application_train_all_models.csv')
print('Обновлённый датасет сохранён.')